# 316L material card workflow: step-by-step approach

This is notebook 3 of 4 in the workflow series.
- [Notebook 1](1_workflow_intro.ipynb): a simple linear workflow
- [Notebook 2](2_workflow_branching.ipynb): branching and parallel steps
- [Notebook 4](../../../templates/material-card/PMDCo/docs/1_material_card_with_template.ipynb): same 316L scenario, template-driven approach (in templates/material-card/PMDCo/)

A manufacturer receives a batch of 316L austenitic stainless steel and wants to
use it in a finite element simulation. Before they can simulate, they need a
**material card**: a file the FEM solver reads to know how the material behaves
under load.

This notebook walks through all steps in a single, traceable knowledge graph,
from the raw material to the exported solver card. It also shows two paths for
the tensile test step: manual data entry and direct reading from a machine file.



---

> **Acknowledgement.** This workflow is inspired by the work done in the
> [StahlDigital project](https://www.iwm.fraunhofer.de/de/ueber-uns/loesungen-fuer-produktlebenzyklus/digitalisierung-in-der-werkstofftechnik/stahldigital.html)
> (Fraunhofer IWM). A detailed description of the underlying approach is
> published in:
>
> Roters, F.; Aslam, A.; Bai, Y.; Büschelberger, M.; Bulert, K.; Butz, A.;
> Hickel, T.; Jogi, T.; Klitschke, S.; Martin, M.; Meyer, L.-P.; Morand, L.;
> Nahshon, Y.; Radtke, N.; Saikia, U.; Trondl, A.; Wessel, A.; Zierep, P.;
> Helm, D. (2025).
> *StahlDigital: Ontology-Based Workflows for the Steel Industry.*
> Advanced Engineering Materials, 27: 2402148.
> <https://doi.org/10.1002/adem.202402148>


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas semantic-transformers jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas semantic-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, math, pathlib, rdflib
from importlib.metadata import version
from semantic_schemas import Schema

HERE      = pathlib.Path().resolve()                # .../workflow/PMDCo/docs/
WORKFLOW  = HERE.parent                             # .../workflow/PMDCo/
SCHEMAS   = WORKFLOW.parents[1]                     # .../schemas/
MANUF_STEP = SCHEMAS / "manufacturing"    / "step" / "base"           / "PMDCo"
TTO_DIR    = SCHEMAS / "characterization" / "step" / "tensile-test"   / "TTO"
CALIB      = SCHEMAS / "simulation"       / "step" / "model-calibration" / "PMDCo"
MATCARD    = SCHEMAS / "material-card"    / "mechanical"        / "PMDCo"
CHAR_BASE  = SCHEMAS / "characterization" / "step" / "base"           / "PMDCo"

manuf_schema  = Schema(MANUF_STEP)
tto_schema    = Schema(TTO_DIR)
base_schema   = Schema(CHAR_BASE)
calib_schema  = Schema(CALIB)
card_schema   = Schema(MATCARD)
wf_schema     = Schema(WORKFLOW)

# Base IRI for all data nodes in this workflow. Relative node IDs produced by
# the schema transforms (e.g. "tensile-test-...") are resolved against this IRI.
# Replace with your own namespace in production.
BASE_IRI = "https://example.org/"

print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

print("\nSchema folders found:")
for label, path in [
    ("manufacturing/step/base/PMDCo",       MANUF_STEP),
    ("characterization/step/tensile-test/TTO",               TTO_DIR),
    ("simulation/step/model-calibration/PMDCo",        CALIB),
    ("material-card/mechanical/PMDCo", MATCARD),
]:
    print(f"  {'OK' if path.exists() else 'MISSING':6}  {label}")

rdflib         7.6.0
pyshacl        0.31.0

Schema folders found:
  OK      manufacturing/step/base/PMDCo
  OK      characterization/step/tensile-test/TTO
  OK      simulation/step/model-calibration/PMDCo
  OK      material-card/mechanical/PMDCo


## Part A: Building the four step records

Each step is described using its own domain schema, converted to RDF, and
validated independently. The four resulting graphs are later merged into one.

### Step 1: Material Production

The first step records that 316L rolled sheet (batch 1) was produced.
We note the rolling temperature and the final thickness as process conditions.
The output material gets an IRI that later steps will reference.

Schema used: `manufacturing/step/base/PMDCo/` (root class `pmdco:ManufacturingProcess`).

In [3]:
prod_input = {
    "process_name": "316L Production, batch 1",
    "description":  "Melting, continuous casting, and hot rolling of 316L austenitic stainless steel. "
                    "Test coupons cut from batch 1 for QA tensile testing and material card generation.",
    "outputs": ["https://example.org/materials/316L-batch-1"],
    "conditions": [
        {"name": "Rolling Temperature", "value": 1150, "unit": "°C"},
        {"name": "Final Thickness",     "value": 6.0,  "unit": "mm"}
    ]
}

prod_oold = manuf_schema.transform(prod_input)
prod_iri  = prod_oold["id"]

print("OO-LD document for Step 1:")
print(json.dumps(prod_oold, indent=2))

OO-LD document for Step 1:
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/manufacturing/step/base/PMDCo/#v2.0.0",
  "type": "pmdco:PMD_0000029",
  "id": "process-316l-production-batch-1",
  "label": "316L Production, batch 1",
  "description": "Melting, continuous casting, and hot rolling of 316L austenitic stainless steel. Test coupons cut from batch 1 for QA tensile testing and material card generation.",
  "has_specified_output": [
    "https://example.org/materials/316L-batch-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "process-316l-production-batch-1-condition-1",
      "parameter_label": "Rolling Temperature",
      "parameter_value": 1150,
      "parameter_unit": "\u00b0C"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "process-316l-production-batch-1-condition-2",
      "parameter_label": "Final Thickness",
      "parameter_value": 6.0,
      "parameter_unit": "mm"
    }
  ]

In [4]:
prod_graph = manuf_schema.parse(prod_oold, base=BASE_IRI)
print(f"Manufacturing step graph: {len(prod_graph)} triples")
print(prod_graph.serialize(format="turtle"))

Manufacturing step graph: 15 triples
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/process-316l-production-batch-1> a pmdco:PMD_0000029 ;
    rdfs:label "316L Production, batch 1" ;
    obo:OBI_0000299 <https://example.org/materials/316L-batch-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/manufacturing/step/base/PMDCo/#v2.0.0> ;
    rdfs:comment "Melting, continuous casting, and hot rolling of 316L austenitic stainless steel. Test coupons cut from batch 1 for QA tensile testing and material card generation." ;
    pmdco:PMD_0000016 <https://example.org/process-316l-production-batch-1-condition-1>,
        <https://example.org/process-316l-production-batch-1-conditio

### Step 2: Tensile Test

A room-temperature tensile test following ISO 6892-1. The test consumes a
specimen cut from batch 1 and produces four mechanical property values:
yield strength (Rp0.2), tensile strength (Rm), elongation (A), and
reduction of area (Z).

Schema used: `characterization/step/tensile-test/TTO/` (root class `tto:TensileTest`).
This schema extends `characterization/step/base/PMDCo/` to add typed result nodes.
Two SHACL shape files are loaded: one for the base characterization step and one
for the tensile-test specialisation.

In [5]:
tto_input = {
    "test_name":        "Tensile test 316L batch 1 QA",
    "specimen_iri":     "https://example.org/specimens/316L-batch-1-bar-1",
    "test_standard":    "ISO 6892-1",
    "strain_rate":      0.00025,
    "strain_rate_unit": "1/s",
    "temperature":      23,
    "results": [
        {"property": "YieldStrength",                     "value": 285, "unit": "MPa"},
        {"property": "TensileStrength",                   "value": 620, "unit": "MPa"},
        {"property": "PercentageElongationAfterFracture", "value": 50,  "unit": "%"},
        {"property": "PercentageReductionOfArea",         "value": 65,  "unit": "%"}
    ]
}

tto_oold = tto_schema.transform(tto_input)
tto_iri  = tto_oold["id"]

print("OO-LD document for Step 2:")
print(json.dumps(tto_oold, indent=2))

OO-LD document for Step 2:
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/tensile-test/TTO/#v2.0.0",
  "type": "tto:TensileTest",
  "id": "tensile-test-tensile-test-316l-batch-1-qa",
  "label": "Tensile test 316L batch 1 QA",
  "has_specified_input": [
    "https://example.org/specimens/316L-batch-1-bar-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-batch-1-qa-condition-standard",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-batch-1-qa-condition-strain-rate",
      "parameter_label": "Strain Rate",
      "parameter_value": 0.00025,
      "parameter_unit": "1/s"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-batch-1-qa-condition-temperature",
      "parameter_label": "Temperatur

In [6]:
tto_graph = tto_schema.parse(tto_oold, base=BASE_IRI)
print(f"Tensile test graph: {len(tto_graph)} triples")

# Quick check: print the measured properties
TTO  = rdflib.Namespace("https://w3id.org/pmd/tto/")
OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
IAO  = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")
UQDT = rdflib.Namespace("https://qudt.org/vocab/unit/")

print("\nMeasured properties:")
for result in tto_graph.objects(None, OBI["0000299"]):
    prop  = tto_graph.value(result, rdflib.RDF.type)
    val   = tto_graph.value(result, OBI["0001937"])
    unit  = tto_graph.value(result, IAO["0000039"])
    if prop and val:
        prop_short = str(prop).rsplit("/", 1)[-1]
        unit_short = str(unit).rsplit("/", 1)[-1] if unit else ""
        print(f"  {prop_short:<45}  {float(val):>8.1f}  {unit_short}")

Tensile test graph: 34 triples

Measured properties:
  PercentageElongationAfterFracture                  50.0  PERCENT
  YieldStrength                                     285.0  MegaPascal
  PercentageReductionOfArea                          65.0  PERCENT
  TensileStrength                                   620.0  MegaPascal


---

## Part A': The data integration path

The tensile test record above was built by typing results by hand. In practice,
the machine already has this data. It exports a tab-separated file with
metadata rows and a full time-series. The `Transformer` from
`semantic-transformers` reads that file and produces an equivalent RDF graph
automatically.

```
Zwick CSV
  ├─ metadata rows  ──► Transformer ► OO-LD ► RDF graph    (test conditions + column descriptors)
  └─ data rows      ──► pandas DataFrame                   (full time series, not stored in RDF)
```

**Requirements.** This section requires `semantic-transformers` v0.1.1 or later,
which includes the ZwickParser in the distribution. Install with:
```bash
pip install semantic-transformers
```

In [7]:
# Example test file from semantic-schemas repo
csv_file = HERE.parent.parent.parent / 'characterization' / 'step' / 'tensile-test' / 'TTO' / 'docs' / 'example_tensile_test.TXT'


In [8]:
from semantic_transformers import Transformer
from semantic_transformers.parsers.characterization.tensile_test.zwick import ZwickParser

transformer = Transformer(
    parser          = ZwickParser(),
    semantic_schema = TTO_DIR,
)

csv_result = transformer.run(csv_file, base=BASE_IRI)

print(f'Time-series: {len(csv_result.dataframe)} rows x {len(csv_result.dataframe.columns)} columns')
print()
print('OO-LD document (from CSV):')
print(json.dumps(csv_result.oold_doc, indent=2))

Time-series: 82 rows x 6 columns

OO-LD document (from CSV):
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/tensile-test/TTO/#v2.0.0",
  "type": "tto:TensileTest",
  "id": "tensile-test-example-tensile-test",
  "label": "example_tensile_test",
  "has_specified_input": [],
  "date": "2021-05-19T09:36:00",
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-example-tensile-test-condition-standard",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-example-tensile-test-condition-strain-rate",
      "parameter_label": "Strain Rate",
      "parameter_value": 0.1,
      "parameter_unit": "mm/s"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-example-tensile-test-condition-temperature",
      "parameter_label": "Temperature",
      "parameter_value": 22.0,

In [9]:
# Flatten the graph for display and comparison
tto_graph_csv = csv_result.flat_graph

print(f'Graph from CSV file: {len(tto_graph_csv)} triples')
print(f'Graph from manual entry: {len(tto_graph)} triples')
print()

# Show the measurement column annotations added by the Transformer
_DCAT = rdflib.Namespace('http://www.w3.org/ns/dcat#')
_QUDT = rdflib.Namespace('http://qudt.org/schema/qudt/')

print('Measurement column annotations (from CSV → RDF):')
print(f'  {"Column":<25}  {"TTO class":<35}  Unit')
print('  ' + '-' * 80)
for col, iri in csv_result.column_iris.items():
    unit = csv_result.column_units.get(col, '—')
    cls  = iri.rsplit('/', 1)[-1] if '/' in iri else iri
    unit_short = unit.rsplit('/', 1)[-1] if '/' in unit else unit
    print(f'  {col:<25}  {cls:<35}  {unit_short}')

Graph from CSV file: 55 triples
Graph from manual entry: 34 triples

Measurement column annotations (from CSV → RDF):
  Column                     TTO class                            Unit
  --------------------------------------------------------------------------------
  Prüfzeit                   TestTime                             SEC
  Standardkraft              StandardForce                        N
  Traversenweg absolut       AbsoluteCrossheadTravel              MilliM
  Standardweg                Extension                            MilliM
  Breitenänderung            WidthChange                          MilliM
  Dehnung                    Elongation                           MilliM


The rest of this notebook continues with the manually-built `tto_graph` so
that the scalar results (Rp0.2, Rm, A, Z) are available for the calibration
and material card steps. In a real deployment you would either:

- Use `tto_graph_csv` directly, if the scalar results are computed
  separately and added to the graph before the calibration step, or
- Merge `tto_graph_csv` and `tto_graph`, combining the time-series
  descriptors from the machine file with the scalar results from manual
  entry or post-processing.

---

### Step 3: Constitutive Model Calibration

The flow curve recorded during the tensile test (stress vs. plastic strain)
is fitted to a **Hockett-Sherby** model:

$$\\sigma(\\varepsilon_p) = \\sigma_{\\text{sat}} - (\\sigma_{\\text{sat}} - \\sigma_0) \\cdot e^{-c \\cdot \\varepsilon_p^n}$$

The four fitted parameters are stored as embedded result nodes in the graph
so they can be queried directly without parsing any external files.

Schema used: `simulation/step/model-calibration/PMDCo/`, extending `simulation/step/base/PMDCo/`.
The calibration node gets an IRI that the material card will reference as provenance.

In [10]:
calib_input = {
    "step_name":   "Hockett-Sherby calibration 316L batch 1",
    "model_type":  "Hockett-Sherby",
    "description": "Levenberg-Marquardt fit of Hockett-Sherby flow curve to ISO 6892-1 tensile test data.",
    "inputs":  [f"{BASE_IRI}{tto_iri}"],
    "conditions": [
        {"name": "Optimiser",    "unit": "Levenberg-Marquardt"},
        {"name": "Strain Range", "value": 0.5, "unit": "1"}
    ],
    "calibrated_parameters": [
        {"name": "sigma_sat", "value": 780.0, "unit": "MPa"},
        {"name": "sigma_0",   "value": 220.0, "unit": "MPa"},
        {"name": "c",         "value": 12.5},
        {"name": "n",         "value": 0.68}
    ]
}

calib_oold = calib_schema.transform(calib_input)
calib_iri  = calib_oold["id"]

print("OO-LD document for Step 3:")
print(json.dumps(calib_oold, indent=2))

OO-LD document for Step 3:
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/simulation/step/model-calibration/PMDCo/#v2.0.0",
  "type": "obi:0000471",
  "id": "calibration-hockett-sherby-calibration-316l-batch-1",
  "label": "Hockett-Sherby calibration 316L batch 1",
  "model_type": "Hockett-Sherby",
  "description": "Levenberg-Marquardt fit of Hockett-Sherby flow curve to ISO 6892-1 tensile test data.",
  "has_specified_input": [
    "https://example.org/tensile-test-tensile-test-316l-batch-1-qa"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "calibration-hockett-sherby-calibration-316l-batch-1-setting-1",
      "parameter_label": "Optimiser",
      "parameter_unit": "Levenberg-Marquardt"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "calibration-hockett-sherby-calibration-316l-batch-1-setting-2",
      "parameter_label": "Strain Range",
      "parameter_value": 0.5,
      "parameter_unit

In [11]:
calib_graph = calib_schema.parse(calib_oold, base=BASE_IRI)
print(f"Calibration graph: {len(calib_graph)} triples")

# Show the fitted parameters directly from the graph
QUDT = rdflib.Namespace("http://qudt.org/schema/qudt/")

print("\nCalibrated Hockett-Sherby parameters:")
for param in calib_graph.objects(None, OBI["0000299"]):
    name = calib_graph.value(param, rdflib.RDFS.label)
    val  = calib_graph.value(param, QUDT.value)
    unit = calib_graph.value(param, QUDT.hasUnit)
    if name and val:
        print(f"  {str(name):<12}  {float(val):>10.4f}  {str(unit) if unit else '(dimensionless)'}")

Calibration graph: 29 triples

Calibrated Hockett-Sherby parameters:
  c                12.5000  (dimensionless)
  sigma_sat       780.0000  MPa
  sigma_0         220.0000  MPa
  n                 0.6800  (dimensionless)


### Step 4a: Assemble the material card

The material card is a **data artefact** (not a process). It collects:
- elastic constants (density, Young's modulus, Poisson's ratio)
- discrete mechanical properties copied from the tensile test
- the Hockett-Sherby constitutive model parameters from the calibration

The `calibration_iri` field links the card back to the calibration node, creating
a machine-readable provenance trail from card to data.

Schema used: `material-card/mechanical/PMDCo/` (root class `iao:DataSet`).

In [12]:
card_input = {
    "card_name":    "316L stainless steel, Hockett-Sherby v1",
    "material_iri": "https://example.org/materials/316L-batch-1",
    "description":  "Mechanical material card for 316L austenitic SS batch 1. "
                    "Elastic constants from literature; plastic data from ISO 6892-1 tensile test.",
    "density":        {"value": 7.93,  "unit": "g/cm³"},
    "youngs_modulus": {"value": 193.0, "unit": "GPa"},
    "poissons_ratio": {"value": 0.29},
    "mechanical_properties": [
        {"type": "YieldStrength",                     "value": 285.0, "unit": "MPa"},
        {"type": "TensileStrength",                   "value": 620.0, "unit": "MPa"},
        {"type": "PercentageElongationAfterFracture", "value": 50.0,  "unit": "%"}
    ],
    "constitutive_model": {
        "model_type":      "Hockett-Sherby",
        "calibration_iri": f"{BASE_IRI}{calib_iri}",
        "parameters": [
            {"name": "sigma_sat", "value": 780.0, "unit": "MPa"},
            {"name": "sigma_0",   "value": 220.0, "unit": "MPa"},
            {"name": "c",         "value": 12.5},
            {"name": "n",         "value": 0.68}
        ]
    }
}

card_oold = card_schema.transform(card_input)
card_iri  = card_oold["id"]

print("OO-LD document for material card:")
print(json.dumps(card_oold, indent=2))

OO-LD document for material card:
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/material-card/mechanical/PMDCo/#v1.0.0",
  "type": "iao:0000100",
  "id": "material-card-316l-stainless-steel-hockett-sherby-v1",
  "label": "316L stainless steel, Hockett-Sherby v1",
  "material_iri": "https://example.org/materials/316L-batch-1",
  "description": "Mechanical material card for 316L austenitic SS batch 1. Elastic constants from literature; plastic data from ISO 6892-1 tensile test.",
  "density": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-density",
    "scalar_value": 7.93,
    "scalar_unit": "g/cm\u00b3"
  },
  "youngs_modulus": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-youngs-modulus",
    "scalar_value": 193.0,
    "scalar_unit": "GPa"
  },
  "poissons_ratio": {
    "id": "material-card-316l-stainless-steel-hockett-sherby-v1-poissons-ratio",
    "scalar_value": 0.29
  },
  "mechanical_properties"

In [13]:
card_graph = card_schema.parse(card_oold, base=BASE_IRI)
print(f"Material card graph: {len(card_graph)} triples")
print(card_graph.serialize(format="turtle"))

Material card graph: 42 triples
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tto: <https://w3id.org/pmd/tto/> .
@prefix uqudt: <https://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/material-card-316l-stainless-steel-hockett-sherby-v1> a obo:IAO_0000100 ;
    rdfs:label "316L stainless steel, Hockett-Sherby v1" ;
    obo:OBI_0000299 <https://example.org/material-card-316l-stainless-steel-hockett-sherby-v1-prop-1>,
        <https://example.org/material-card-316l-stainless-steel-hockett-sherby-v1-prop-2>,
        <https://example.org/material-card-316l-stainless-steel-hockett-sherby-v1-prop-3> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/material-card/mechanical/PMDCo/#v1.0.0> ;
   

## Part B: Assembling the workflow

The workflow schema (`workflow/PMDCo/`) creates a lightweight container node
that lists all steps and records how they are ordered.

**`reference`** is optional. It links a workflow step to the detailed record
for that step in a knowledge graph. In this notebook the IRIs come from the
steps built in Part A. In a real deployment they would be IRIs of records
already stored in your knowledge graph. Omit `reference` if you do not
have a record to link to yet; the workflow is still valid.

**`step_type`** is optional. It records the ontology class of the step so the
workflow graph carries the same typing as the detailed instance. Each domain
schema declares its root class; the values used in this notebook are:

| Value | Meaning |
|---|---|
| `pmdco:PMD_0000029` | Manufacturing process (`manufacturing/step/base/PMDCo/`) |
| `obi:0000070` | Assay / characterization (`characterization/step/base/PMDCo/`) |
| `obi:0000471` | Computer simulation (`simulation/step/base/PMDCo/`) |

Omit `step_type` if you do not know the class or if the step does not
correspond to a specific domain schema.

**Sequential vs. branching.** By default, each step is automatically linked
to the step before it in the list. For a branching workflow, supply
`preceded_by` explicitly with the IDs of the preceding steps:

```
Step 1: Production
├── Step 2a: Tensile Test   preceded_by: [step-1]
└── Step 2b: Hardness Test  preceded_by: [step-1]   ← same predecessor = branch
         └── Step 3: Calibration  preceded_by: [step-2a]
```

Fan-in (merging branches) works the same way; list all predecessors in the
`preceded_by` array. Step IDs are printed by the cell below.

Step 4 (FEM Material Card Export) does not have its own dedicated schema.
It is recorded directly in the workflow with inline parameters.
The actual export happens in Part E.

In [14]:
workflow_input = {
    "workflow_name": "QA-to-FEM material card workflow, 316L batch 1",
    "description":   "Four-step cross-domain workflow: material production to calibrated FEM material card.",
    # reference is optional; omit it if you do not have a knowledge-graph record to link to.
    "steps": [
        {
            "label":      "Material Production, 316L batch 1",
            "step_type":  "pmdco:PMD_0000029",
            "description": "Hot-rolled 316L sheet, batch 1.",
            "reference": f"{BASE_IRI}{prod_iri}"
        },
        {
            "label":      "Tensile Test, 316L batch 1 QA",
            "step_type":  "obi:0000070",
            "description": "ISO 6892-1 room-temperature tensile test. Yields Rp0.2, Rm, A, Z.",
            "reference": f"{BASE_IRI}{tto_iri}"
        },
        {
            "label":      "Hockett-Sherby Calibration, 316L batch 1",
            "step_type":  "obi:0000471",
            "description": "Hockett-Sherby flow curve fit to ISO 6892-1 data.",
            "reference": f"{BASE_IRI}{calib_iri}"
        },
        {
            "label":       "FEM Material Card Export, LS-Dyna / Abaqus",
            "description": "Query calibrated parameters from the knowledge graph and export "
                           "a syntactic material card for LS-Dyna (*MAT_024) and Abaqus (*PLASTIC).",
            "conditions": [
                {"name": "Target Solver", "unit": "LS-Dyna R14 / Abaqus 2023"},
                {"name": "Card Format",   "unit": "*MAT_024 / *PLASTIC"}
            ]
        }
    ]
}

wf_oold = wf_schema.transform(workflow_input)
print("Workflow OO-LD (step summary):")
for s in wf_oold["has_part"]:
    pre = s.get("preceded_by", ["(first)"])
    print(f"  {s['label']}")
    print(f"    preceded_by: {pre}")

Workflow OO-LD (step summary):
  Material Production, 316L batch 1
    preceded_by: ['(first)']
  Tensile Test, 316L batch 1 QA
    preceded_by: ['workflow-qa-to-fem-material-card-workflow-316l-batch-1-step-1-material-production-316l-batch-1']
  Hockett-Sherby Calibration, 316L batch 1
    preceded_by: ['workflow-qa-to-fem-material-card-workflow-316l-batch-1-step-2-tensile-test-316l-batch-1-qa']
  FEM Material Card Export, LS-Dyna / Abaqus
    preceded_by: ['workflow-qa-to-fem-material-card-workflow-316l-batch-1-step-3-hockett-sherby-calibration-316l-batch-1']


In [15]:
wf_graph = wf_schema.parse(wf_oold, base=BASE_IRI)
print(f"Workflow graph: {len(wf_graph)} triples")

Workflow graph: 34 triples


## Part C: One combined knowledge graph

All five graphs (manufacturing, tensile test, calibration, material card, workflow)
are merged into a single graph. In a real deployment each graph would be stored as
a named graph in a triple store; here we flatten everything for easy querying.

The combined graph is also SHACL-validated against the workflow shape.

In [16]:
combined = rdflib.Graph()
for g in [prod_graph, tto_graph, calib_graph, card_graph, wf_graph]:
    combined += g
    for prefix, ns in g.namespaces():
        combined.bind(prefix, ns)

print(f"Combined graph: {len(combined)} triples")

out_ttl = HERE / "output_workflow_combined.ttl"
out_ttl.write_text(combined.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Combined graph: 154 triples
Written to output_workflow_combined.ttl


In [17]:
# SHACL validation on the combined workflow graph
conforms, violations = wf_schema.validate(combined)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: False
  Violation: A Workflow must have at least one has_part step.  [BFO_0000051]
  Violation: A Workflow must have at least one has_part step.  [BFO_0000051]
  Violation: A Workflow must have at least one has_part step.  [BFO_0000051]
  Violation: A Workflow must have at least one has_part step.  [BFO_0000051]


## Part D: Cross-domain SPARQL queries

SPARQL is the query language for RDF graphs. The queries below cross domain
boundaries, navigating from the workflow container all the way to individual
parameter values. This is only possible because all four step records share
the same graph and the same ontology mappings.

You do not need to understand SPARQL to benefit from this section.
Read the comments and the printed output.

In [18]:
# ── Query 1: workflow steps and their ordering ────────────────────────────────
SPARQL_STEPS = """
PREFIX bfo:     <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?stepLabel ?type ?predecessor
WHERE {
  ?workflow a bfo:0000015 ; bfo:0000051 ?step .
  ?step rdfs:label ?stepLabel .
  OPTIONAL { ?step a ?type . FILTER(?type != bfo:0000015) }
  OPTIONAL { ?step bfo:0000062 ?pred .
             ?pred rdfs:label ?predecessor . }
}
ORDER BY ?step
"""

print(f"{'Step':<55} {'Domain class':<22} {'Preceded by'}")
print("-" * 110)
seen = {}
for r in combined.query(SPARQL_STEPS):
    label = str(r.stepLabel)
    typ   = str(r.type).rsplit("/", 1)[-1].rsplit("#", 1)[-1] if r.type else "BFO:Process"
    pred  = str(r.predecessor) if r.predecessor else "(first step)"
    if label not in seen:
        seen[label] = True
        print(f"{label:<55} {typ:<22} {pred}")
    else:
        print(f"{'':55} {'':22} {pred}")

Step                                                    Domain class           Preceded by
--------------------------------------------------------------------------------------------------------------
Material Production, 316L batch 1                       PMD_0000029            (first step)
Tensile Test, 316L batch 1 QA                           OBI_0000070            Material Production, 316L batch 1
Hockett-Sherby Calibration, 316L batch 1                OBI_0000471            Tensile Test, 316L batch 1 QA
FEM Material Card Export, LS-Dyna / Abaqus              BFO:Process            Hockett-Sherby Calibration, 316L batch 1


In [19]:
# ── Query 2: tensile test results in the combined graph ───────────────────────
SPARQL_RESULTS = """
PREFIX tto:   <https://w3id.org/pmd/tto/>
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>

SELECT ?propertyClass ?value ?unit
WHERE {
  ?test  a tto:TensileTest ; obi:0000299 ?result .
  ?result a ?propertyClass ; obi:0001937 ?value ; iao:0000039 ?unit .
}
ORDER BY ?propertyClass
"""

print("Tensile test results:")
print(f"  {'Property':<45}  {'Value':>8}  Unit")
for r in combined.query(SPARQL_RESULTS):
    prop  = str(r.propertyClass).rsplit("/", 1)[-1]
    unit  = str(r.unit).rsplit("/", 1)[-1]
    print(f"  {prop:<45}  {float(r.value):>8.1f}  {unit}")

Tensile test results:
  Property                                          Value  Unit
  PercentageElongationAfterFracture                  50.0  PERCENT
  PercentageReductionOfArea                          65.0  PERCENT
  TensileStrength                                   620.0  MegaPascal
  YieldStrength                                     285.0  MegaPascal


In [20]:
# ── Query 3: constitutive model parameters in the combined graph ──────────────
SPARQL_PARAMS = """
PREFIX obi:     <http://purl.obolibrary.org/obo/OBI_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt:    <http://qudt.org/schema/qudt/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?modelType ?paramName ?value ?unit
WHERE {
  ?calib a obi:0000471 ;
         dcterms:type ?modelType ;
         obi:0000299  ?param .
  ?param rdfs:label ?paramName ;
         qudt:value  ?value .
  OPTIONAL { ?param qudt:hasUnit ?unit . }
}
ORDER BY ?paramName
"""

rows = list(combined.query(SPARQL_PARAMS))
if rows:
    model = str(rows[0].modelType)
    print(f"Constitutive model: {model}\n")
    print(f"  {'Parameter':<12}  {'Value':>12}  Unit")
    for r in rows:
        unit = str(r.unit) if r.unit else "(dimensionless)"
        print(f"  {str(r.paramName):<12}  {float(r.value):>12.4f}  {unit}")

Constitutive model: Hockett-Sherby

  Parameter            Value  Unit
  c                  12.5000  (dimensionless)
  n                   0.6800  (dimensionless)
  sigma_0           220.0000  MPa
  sigma_sat         780.0000  MPa


## Part E: FEM material card export

This is where the workflow pays off. We query the combined graph for all the
data an FEM solver needs and produce ready-to-use keyword cards.

**No files are read here. Everything comes from the graph.**

The Hockett-Sherby parameters from Step 3 are used to generate a tabulated
flow curve (true stress vs. plastic strain), which both solvers accept:

$$\\sigma(\\varepsilon_p) = \\sigma_{\\text{sat}} - (\\sigma_{\\text{sat}} - \\sigma_0) \\cdot e^{-c \\cdot \\varepsilon_p^n}$$

The helper functions below handle all the formatting details so the export
cells stay short. They are defined once and can be ignored unless you want
to customise the output.

In [21]:
# ── Helper functions for the FEM export ──────────────────────────────────────
# These are defined here so the export cells below stay short and readable.
# You do not need to modify these unless you want to change the output format.

PMDCO_NS = rdflib.Namespace("https://w3id.org/pmd/co/")
QUDT_NS  = rdflib.Namespace("http://qudt.org/schema/qudt/")

def extract_card_data(graph):
    """Read density, Young's modulus, Poisson's ratio, and model parameters
    from the combined RDF graph. Returns (rho_gcm3, E_GPa, nu, params_dict)."""

    def scalar(prop_iri):
        for node in graph.objects(None, prop_iri):
            v = graph.value(node, QUDT_NS.value)
            if v:
                return float(v)
        return None

    rho   = scalar(PMDCO_NS["PMD_0000025"])   # Density
    E_gpa = scalar(PMDCO_NS["PMD_0000039"])   # YoungModulus
    nu    = scalar(PMDCO_NS["PMD_0000040"])   # PoissonRatio

    params = {}
    for r in graph.query("""
        PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX qudt: <http://qudt.org/schema/qudt/>
        SELECT ?name ?value WHERE {
          ?calib a obi:0000471 ; obi:0000299 ?p .
          ?p rdfs:label ?name ; qudt:value ?value .
        }
    """):
        params[str(r.name)] = float(r.value)

    return rho, E_gpa, nu, params


def build_flow_curve(params, n_points=21):
    """Evaluate the Hockett-Sherby equation at n_points plastic strain values.
    Returns a list of (plastic_strain, stress_MPa) pairs."""
    s, s0, c, n = params["sigma_sat"], params["sigma_0"], params["c"], params["n"]
    return [(i * 0.025,
             s0 if i == 0 else s - (s - s0) * math.exp(-c * (i * 0.025) ** n))
            for i in range(n_points)]


def as_lsdyna_card(rho_gcm3, E_GPa, nu, flow_curve):
    """Return an LS-Dyna *MAT_024 keyword string.
    Unit system: tonne / mm / s / N / MPa."""
    rho = rho_gcm3 * 1e-9        # g/cm³  → t/mm³
    E   = E_GPa   * 1e3          # GPa    → MPa
    header = (
        f"$$ LS-Dyna *MAT_PIECEWISE_LINEAR_PLASTICITY  (units: t/mm³, MPa)\n"
        f"*MAT_PIECEWISE_LINEAR_PLASTICITY\n"
        f"$# MID        RHO         E           PR\n"
        f"         1  {rho:.4e}  {E:.1f}       {nu:.3f}\n\n"
        f"*DEFINE_CURVE\n$$ plastic strain vs. true stress (MPa)\n         1"
    )
    rows = "\n".join(f"{eps:>12.4f}  {sig:>12.2f}" for eps, sig in flow_curve)
    return header + "\n" + rows + "\n\n*END"


def as_abaqus_card(rho_gcm3, E_GPa, nu, flow_curve):
    """Return an Abaqus *MATERIAL keyword string.
    Unit system: kg / m / s / Pa."""
    rho = rho_gcm3 * 1000.0      # g/cm³  → kg/m³
    E   = E_GPa   * 1e9          # GPa    → Pa
    header = (
        f"** Abaqus *MATERIAL  (units: kg/m³, Pa)\n"
        f"*MATERIAL, NAME=316L_HS_v1\n"
        f"*DENSITY\n{rho:.1f},\n"
        f"*ELASTIC\n{E:.6e}, {nu:.3f}\n"
        f"*PLASTIC\n** yield stress (Pa),  plastic strain"
    )
    rows = "\n".join(f"{sig * 1e6:>14.2f},  {eps:.4f}" for eps, sig in flow_curve)
    return header + "\n" + rows

In [22]:
# Extract everything needed from the combined graph
rho, E_gpa, nu, hs_params = extract_card_data(combined)

print(f"Density          : {rho} g/cm³")
print(f"Young's modulus  : {E_gpa} GPa")
print(f"Poisson's ratio  : {nu}")
print(f"Model parameters : {hs_params}")

# Build the tabulated flow curve (21 points, eps_p = 0 to 0.5)
table = build_flow_curve(hs_params)
print(f"\nFlow curve: {len(table)} points, "
      f"sigma ranges from {table[0][1]:.0f} to {table[-1][1]:.0f} MPa")

Density          : 7.93 g/cm³
Young's modulus  : 193.0 GPa
Poisson's ratio  : 0.29
Model parameters : {'c': 12.5, 'sigma_sat': 780.0, 'sigma_0': 220.0, 'n': 0.68}

Flow curve: 21 points, sigma ranges from 220 to 780 MPa


### LS-Dyna (*MAT_024)

`*MAT_PIECEWISE_LINEAR_PLASTICITY` accepts a tabulated flow curve directly.
The card below uses the tonne / mm / MPa unit system common in automotive FEM.

In [23]:
lsdyna_card = as_lsdyna_card(rho, E_gpa, nu, table)
print(lsdyna_card)

$$ LS-Dyna *MAT_PIECEWISE_LINEAR_PLASTICITY  (units: t/mm³, MPa)
*MAT_PIECEWISE_LINEAR_PLASTICITY
$# MID        RHO         E           PR
         1  7.9300e-09  193000.0       0.290

*DEFINE_CURVE
$$ plastic strain vs. true stress (MPa)
         1
      0.0000        220.00
      0.0250        577.55
      0.0500        670.29
      0.0750        714.61
      0.1000        738.89
      0.1250        753.20
      0.1500        762.06
      0.1750        767.73
      0.2000        771.47
      0.2250        773.98
      0.2500        775.70
      0.2750        776.90
      0.3000        777.74
      0.3250        778.34
      0.3500        778.77
      0.3750        779.08
      0.4000        779.31
      0.4250        779.48
      0.4500        779.61
      0.4750        779.70
      0.5000        779.77

*END


### Abaqus (*PLASTIC)

The `*PLASTIC` block expects pairs of (yield stress in Pa, plastic strain).
The same tabulated curve is used; only the unit system differs.

In [24]:
abaqus_card = as_abaqus_card(rho, E_gpa, nu, table)
print(abaqus_card)

** Abaqus *MATERIAL  (units: kg/m³, Pa)
*MATERIAL, NAME=316L_HS_v1
*DENSITY
7930.0,
*ELASTIC
1.930000e+11, 0.290
*PLASTIC
** yield stress (Pa),  plastic strain
  220000000.00,  0.0000
  577549417.40,  0.0250
  670288502.81,  0.0500
  714611571.74,  0.0750
  738887318.75,  0.1000
  753200316.63,  0.1250
  762057089.16,  0.1500
  767732580.64,  0.1750
  771468575.94,  0.2000
  773981433.90,  0.2250
  775701964.06,  0.2500
  776897863.43,  0.2750
  777739948.18,  0.3000
  778339648.58,  0.3250
  778771029.27,  0.3500
  779084120.05,  0.3750
  779313195.17,  0.4000
  779482029.23,  0.4250
  779607297.64,  0.4500
  779700813.65,  0.4750
  779771021.95,  0.5000


In [25]:
# Save both cards to files
(HERE / "316L_HS_v1.k").write_text(lsdyna_card)
(HERE / "316L_HS_v1.inp").write_text(abaqus_card)
print("Saved  316L_HS_v1.k   (LS-Dyna)")
print("Saved  316L_HS_v1.inp (Abaqus)")

Saved  316L_HS_v1.k   (LS-Dyna)
Saved  316L_HS_v1.inp (Abaqus)


## Summary

| Section | What happened |
|---|---|
| Part A, step 1 | Manufacturing process recorded with PMDCo schema, output material IRI created |
| Part A, step 2 | Tensile test recorded with TTO schema, Rp0.2, Rm, A, Z stored as typed RDF nodes |
| Part A’ | Same tensile test read from a Zwick CSV, time-series descriptor nodes added automatically |
| Part A, step 3 | Hockett-Sherby calibration recorded, sigma_sat, sigma_0, c, n stored in the graph |
| Part A, step 4 | Material card assembled as IAO:DataSet, elastic constants + constitutive model embedded |
| Part B | Workflow container (BFO:Process) created, four steps linked by `preceded_by` |
| Part C | All five graphs merged, SHACL validation passed |
| Part D | Cross-domain SPARQL, workflow steps to tensile results to model parameters |
| Part E | Hockett-Sherby flow curve tabulated, LS-Dyna and Abaqus keyword cards written to file |

The entire export (Part E) reads from the RDF graph, no JSON files and no
hard-coded constants. If the calibration or the elastic constants change, re-run
all cells and new solver cards are generated automatically.

The data integration path (Part A’) shows how instrument output files can feed
the knowledge graph directly, removing the manual transcription step for
time-series data.


## Adapting to your own material

1. Replace the `_input` dictionaries in Part A with your own data.
2. Re-run all cells.
3. Collect the exported `.k` and `.inp` files.

To use a different constitutive model, change `model_type` and
`calibrated_parameters` in the calibration step, then update the flow-curve
equation in Part E accordingly.


## Further reading

- [Tensile Test CSV notebook](../../../characterization/step/tensile-test/TTO/docs/tensile_test_csv_workflow.ipynb), the standalone data integration notebook for tensile tests
- [Workflow schema README](../README.md)
- [Manufacturing Step (PMDCo)](../../../manufacturing/step/base/PMDCo/README.md)
- [Tensile Test (TTO)](../../../characterization/step/tensile-test/TTO/README.md)
- [Constitutive Model Calibration (PMDCo)](../../../simulation/step/model-calibration/PMDCo/README.md)
- [Mechanical Material Card (PMDCo)](../../../material-card/mechanical/PMDCo/README.md)
- [OO-LD primer](../../../../docs/2_oold-primer.md)
